# Notebook 02 — Photograph Validation Pipeline

End-to-end pipeline that validates and processes a portrait photograph:

| Step | Description |
|------|-------------|
| 1 | **Face count check** — MTCNN; error if 0 or >1 face detected |
| 2 | **Auto-correct orientation** — coarse 0/90/180/270° + fine tilt (disabled by default) |
| 3 | **Detect & crop face** — MTCNN bounding-box with dynamic margins |
| 4 | **Blur detection** — Laplacian variance; stops pipeline if image is too blurry |
| 5 | **Resize** — proportional scale with white-background composite |
| 6 | **Remove background** — rembg `birefnet-portrait` model with alpha feathering |

## Usage
1. Install dependencies: `uv pip install "rembg[cpu]" mtcnn tensorflow opencv-python Pillow`
2. Place your image under `../data/input/` and set `IMAGE_NAME` below.
3. Run all cells — intermediate and final outputs are saved to `../data/output/pipeline/`.

In [ ]:
# !uv pip install "rembg[cpu]" mtcnn tensorflow opencv-python Pillow
# Uncomment the line above to install dependencies in-notebook.

In [ ]:
import os
import time

# ── CONFIG ────────────────────────────────────────────────────────────────────
IMAGE_NAME = "sample_001.jpg"   # ← change to your image filename
INPUT_DIR  = "../data/input"
INPUT_PATH = os.path.join(INPUT_DIR, IMAGE_NAME)

OUT_ROOT   = "../data/output/pipeline"
CROP_DIR   = os.path.join(OUT_ROOT, "face_crop")
RESIZE_DIR = os.path.join(OUT_ROOT, "resize")
FINAL_DIR  = os.path.join(OUT_ROOT, "remove_bg")

for d in (CROP_DIR, RESIZE_DIR, FINAL_DIR):
    os.makedirs(d, exist_ok=True)

print(f"Input  : {INPUT_PATH}")
print(f"Crop   : {CROP_DIR}")
print(f"Resize : {RESIZE_DIR}")
print(f"Final  : {FINAL_DIR}")

In [ ]:
from PIL import Image
from rembg import remove, new_session

# Initialise rembg session once and warm it up.
# Switch to new_session("birefnet-portrait", providers=["CUDAExecutionProvider"])
# for GPU inference.
session = new_session("birefnet-portrait")
remove(Image.new("RGB", (512, 512)), session=session)   # warmup
print("rembg session ready.")

In [ ]:
from mtcnn import MTCNN
detector = MTCNN()
print("MTCNN detector ready.")

In [ ]:
# Pipeline-wide timing dict
_pipeline_start = time.perf_counter()
pipeline_times  = {}

### Step 1 — Face Count Check (MTCNN)

In [ ]:
import cv2
import matplotlib.pyplot as plt

def check_multiple_faces(image_path, show_on_error=True):
    """Returns 'OK', 'NO_FACE_ERROR', or 'MULTIPLE_FACES_ERROR'.
    When multiple faces are found and *show_on_error* is True, draws
    a red bounding box around each detected face.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results  = detector.detect_faces(img_rgb)
    n        = len(results)

    if n == 0:
        return "NO_FACE_ERROR"

    if n > 1 and show_on_error:
        vis = img_rgb.copy()
        for face in results:
            x, y, w, h = face["box"]
            cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 0, 0), 2)
        plt.figure(figsize=(6, 6))
        plt.imshow(vis)
        plt.title(f"Multiple Faces Detected: {n}")
        plt.axis("off")
        plt.show()

    return "MULTIPLE_FACES_ERROR" if n > 1 else "OK"

_t0          = time.perf_counter()
face_status  = check_multiple_faces(INPUT_PATH)
_elapsed     = time.perf_counter() - _t0
pipeline_times["Step 1 - Face Count Check"] = _elapsed
print(f"Face check result : {face_status}")
print(f"⏱  Step 1 — Face Count Check: {_elapsed:.2f}s")

if face_status != "OK":
    raise RuntimeError(
        f"Pipeline stopped — {face_status}: "
        "Expected exactly one face in the image."
    )

### Step 2 — Auto-Correct Orientation (disabled by default)

In [ ]:
import numpy as np

# ── Orientation correction helpers ───────────────────────────────────────────
# Full implementation is commented out; the image passes through unchanged.
# To enable:
#   1. Uncomment the helper functions below.
#   2. Replace the last line with: oriented_bgr = auto_correct_orientation(INPUT_PATH)

# def rotate_image(img, angle):
#     (h, w) = img.shape[:2]
#     center = (w / 2, h / 2)
#     M = cv2.getRotationMatrix2D(center, angle, 1.0)
#     cos, sin = abs(M[0, 0]), abs(M[0, 1])
#     new_w = int(h * sin + w * cos)
#     new_h = int(h * cos + w * sin)
#     M[0, 2] += new_w / 2 - center[0]
#     M[1, 2] += new_h / 2 - center[1]
#     return cv2.warpAffine(img, M, (new_w, new_h),
#                           flags=cv2.INTER_LINEAR,
#                           borderMode=cv2.BORDER_REPLICATE)

# def orientation_score(img):
#     """Heuristic score based on face keypoint geometry (higher = more upright)."""
#     faces = detector.detect_faces(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
#     if not faces:
#         return -999
#     kp = faces[0]['keypoints']
#     le, re, nose = kp['left_eye'], kp['right_eye'], kp['nose']
#     score  = -abs(re[1] - le[1])                       # penalise eye tilt
#     score += 100 if nose[1] > (le[1] + re[1]) / 2 else -100  # nose below eyes
#     x, y, bw, bh = faces[0]['box']
#     if bh > bw:  score += 20                           # portrait aspect
#     if le[0] < re[0]: score += 10                     # correct L-R order
#     return score

# def auto_correct_orientation(image_path):
#     img  = cv2.imread(image_path)
#     best, best_score = img, -9999
#     for angle in [0, 90, 180, 270]:
#         rot   = rotate_image(img, angle)
#         score = orientation_score(rot)
#         if score > best_score:
#             best_score, best = score, rot
#     angle = np.degrees(np.arctan2(
#         detector.detect_faces(cv2.cvtColor(best, cv2.COLOR_BGR2RGB))[0]['keypoints']['right_eye'][1]
#         - detector.detect_faces(cv2.cvtColor(best, cv2.COLOR_BGR2RGB))[0]['keypoints']['left_eye'][1],
#         detector.detect_faces(cv2.cvtColor(best, cv2.COLOR_BGR2RGB))[0]['keypoints']['right_eye'][0]
#         - detector.detect_faces(cv2.cvtColor(best, cv2.COLOR_BGR2RGB))[0]['keypoints']['left_eye'][0]
#     ))
#     return rotate_image(best, -angle)

# Step 2 disabled — pass raw image straight through.
oriented_bgr = cv2.imread(INPUT_PATH)
print("Step 2 — orientation correction skipped (disabled).")

### Step 3 — Detect & Crop Face (MTCNN)

In [ ]:
def detect_face_and_crop(img_bgr, face_fill_ratio=0.8):
    """Detects the largest face with MTCNN and returns a cropped PIL image.

    Dynamic margins give a wider horizontal and taller vertical region so
    the resulting crop is a natural portrait, not a tight face box.
    """
    img_rgb     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _     = img_rgb.shape
    results     = detector.detect_faces(img_rgb)
    if not results:
        raise RuntimeError("MTCNN found no face.")

    face         = sorted(results, key=lambda x: x['box'][2] * x['box'][3], reverse=True)[0]
    x, y, bw, bh = face['box']
    cx, cy       = x + bw // 2, y + bh // 2

    x1 = max(0, cx - int(bw * 1.0));  x2 = min(w, cx + int(bw * 1.0))
    y1 = max(0, cy - int(bh * 1.4));  y2 = min(h, cy + int(bh * 1.4))

    return Image.fromarray(img_rgb[y1:y2, x1:x2])

crop_output_path = os.path.join(CROP_DIR, IMAGE_NAME)
_t0              = time.perf_counter()
cropped_img      = detect_face_and_crop(oriented_bgr)
_elapsed         = time.perf_counter() - _t0
pipeline_times["Step 3 - Face Detect & Crop"] = _elapsed
cropped_img.save(crop_output_path)
print(f"Cropped face saved → {crop_output_path}")
print(f"⏱  Step 3 — Face Detect & Crop: {_elapsed:.2f}s")

### Step 4 — Blur Detection

In [ ]:
def detect_blur(image_path: str, threshold: float = 100.0):
    """Detect if an image is blurry using variance of the Laplacian.

    Parameters
    ----------
    image_path : str
        Path to the image file.
    threshold : float
        Images with a Laplacian variance below this value are considered blurry.
        Higher values make the check more sensitive.

    Returns
    -------
    dict with keys ``is_blurry`` (bool) and ``score`` (float).
    """
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Invalid image path: {image_path}")
    gray         = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    lap_var      = cv2.Laplacian(gray, cv2.CV_64F).var()
    return {"is_blurry": lap_var < threshold, "score": lap_var}

_t0         = time.perf_counter()
blur_result = detect_blur(crop_output_path)
_elapsed    = time.perf_counter() - _t0
pipeline_times["Step 4 - Blur Detection"] = _elapsed
print(f"Blur score : {blur_result['score']:.2f} (threshold: 100.0)")
print(f"Is blurry  : {blur_result['is_blurry']}")
print(f"⏱  Step 4 — Blur Detection: {_elapsed:.2f}s")

if blur_result["is_blurry"]:
    raise RuntimeError(
        f"Pipeline stopped — BLUR_ERROR: "
        f"Image too blurry (score={blur_result['score']:.2f}, threshold=100.0)."
    )

### Step 5 — Resize

In [ ]:
def resize_image(rgba, face_fill_ratio=0.9, bg_color=(255, 255, 255)):
    """Scale image by *face_fill_ratio* and composite it onto a white background.

    BILINEAR resampling is used instead of LANCZOS for 2-3× speed improvement.
    """
    new_w, new_h = int(rgba.width * face_fill_ratio), int(rgba.height * face_fill_ratio)
    resized      = rgba.resize((new_w, new_h), Image.BILINEAR)
    bg           = Image.new("RGB", (new_w, new_h), bg_color)
    bg.paste(resized, (0, 0), mask=resized.getchannel("A"))
    return bg

resize_output_path = os.path.join(RESIZE_DIR, IMAGE_NAME)
_t0        = time.perf_counter()
resized_img = resize_image(cropped_img.convert("RGBA"))
_elapsed   = time.perf_counter() - _t0
pipeline_times["Step 5 - Resize"] = _elapsed
resized_img.save(resize_output_path)
print(f"Resized image saved → {resize_output_path}")
print(f"⏱  Step 5 — Resize: {_elapsed:.2f}s")

### Step 6 — Remove Background (rembg birefnet-portrait)

In [ ]:
from PIL import ImageFilter

def remove_bg(img):
    """Remove background with rembg and apply light alpha-edge feathering.

    *alpha_matting* is disabled for speed; re-enable it for finer hair edges.
    """
    rgba = remove(
        img.convert("RGBA"),
        session=session,
        alpha_matting=False,
    )
    # Feather the alpha mask slightly to reduce hard edges.
    a = rgba.getchannel("A").filter(ImageFilter.GaussianBlur(1))
    rgba.putalpha(a)
    return rgba

final_output_path = os.path.join(FINAL_DIR, IMAGE_NAME)
_t0       = time.perf_counter()
final_img = remove_bg(resized_img)
_elapsed  = time.perf_counter() - _t0
pipeline_times["Step 6 - Remove Background"] = _elapsed

# Composite onto white for JPEG-compatible output.
bg = Image.new("RGB", final_img.size, (255, 255, 255))
bg.paste(final_img, mask=final_img.getchannel("A"))
bg.save(final_output_path)

total = sum(pipeline_times.values())
pipeline_times["Total Pipeline"] = total
print(f"Final image saved → {final_output_path}")
print(f"⏱  Step 6 — Remove Background: {_elapsed:.2f}s")
print(f"\n⏱  Total pipeline: {total:.2f}s")

### Result Preview

In [ ]:
import matplotlib.pyplot as plt

stages = [
    ("Original",            Image.open(INPUT_PATH).convert("RGB")),
    ("Face Crop",           Image.open(crop_output_path).convert("RGB")),
    ("Resized",             resized_img),
    ("Background Removed",  bg),
]

fig, axes = plt.subplots(1, len(stages), figsize=(5 * len(stages), 6))
for ax, (title, img) in zip(axes, stages):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("\nPer-step timings:")
for step, t in pipeline_times.items():
    print(f"  {step}: {t:.2f}s")